# <font color="#418FDE" size="6.5" uppercase>**Categorical Encoding**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Encode categorical features using ordinal and one-hot strategies. 
- Handle unknown, infrequent, and redundant categories in practical datasets. 
- Integrate categorical encoders into ColumnTransformer workflows. 


## **1. Basic Categorical Encoders**

### **1.1. Ordinal Encoding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_01.jpg?v=1783982382" width="250">



>* Convert categories into numeric labels
>* Best for categories with natural order

>* Compact when categories have meaningful order
>* Can create false numeric relationships

>* Use domain order for truly ordered categories
>* Keep learned mappings consistent across datasets



In [ ]:
#@title Python Code - Ordinal Encoding

# This example demonstrates ordinal encoding.
# Ordered categories become meaningful numeric labels.
# The output shows learned category mappings.

import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# Create a tiny dataset with a natural category order.
shirt_data = pd.DataFrame(
    {"size": ["small", "large", "medium", "small", "extra large"]}
)

# Validate that the expected column is present.
if "size" not in shirt_data.columns:
    raise ValueError("The size column is required for this example.")

# Give the encoder the domain order explicitly.
size_order = [["small", "medium", "large", "extra large"]]
encoder = OrdinalEncoder(categories=size_order)

# Fit learns the mapping, and transform applies it.
encoded_values = encoder.fit_transform(shirt_data[["size"]])
shirt_data["encoded_size"] = encoded_values.astype(int)

# Build a compact mapping table for display.
mapping = pd.DataFrame(
    {"category": encoder.categories_[0], "encoded_value": [0, 1, 2, 3]}
)

print("Original data with ordinal encoded values:")
print(shirt_data.to_string(index=False))
print("Learned ordinal mapping:")
print(mapping.to_string(index=False))



### **1.2. One Hot Encoding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_02.jpg?v=1783982384" width="250">



>* Creates indicators for unordered categories
>* Avoids false numeric relationships between labels

>* Preserves categories without false numeric order
>* Uses indicators for model-friendly comparisons

>* Best for few unordered categories
>* Many categories add features and redundancy



In [ ]:
#@title Python Code - One Hot Encoding

# This example demonstrates one hot encoding.
# Nominal categories become separate indicator columns.
# The output shows encoded feature names.

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Create a tiny dataset with unordered categories.
orders = pd.DataFrame(
    {
        "payment_method": ["credit card", "cash", "debit card", "cash"],
        "shipping_speed": ["standard", "express", "standard", "overnight"],
    }
)

# Validate that the example has the expected two columns.
if orders.shape != (4, 2):
    raise ValueError("Expected four rows and two categorical columns.")

# Fit the encoder and transform the categorical columns.
encoder = OneHotEncoder(sparse_output=False)
encoded_array = encoder.fit_transform(orders)

# Build a readable table from the encoded NumPy array.
encoded_columns = encoder.get_feature_names_out(orders.columns)
encoded_orders = pd.DataFrame(encoded_array, columns=encoded_columns)

# Show the original categories before encoding.
print("Original categorical data:")
print(orders.head(4).to_string(index=False))

# Show the new indicator columns created by one hot encoding.
print("One hot encoded columns:")
print(encoded_orders.head(4).astype(int).to_string(index=False))



### **1.3. Pandas Categories**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_03.jpg?v=1783982386" width="250">



>* Mark repeated labels as categorical values
>* Clarify valid categories and possible ordering

>* Categories store labels compactly using internal codes
>* Use explicit encoders for modeling features

>* Preserve expected categories across data splits
>* Respect ordered labels; encode deliberately later



In [ ]:
#@title Python Code - Pandas Categories

# Show pandas categories before machine learning encoding.
# Compare labels, category order, and internal codes.
# Notice codes are storage, not final features.

import pandas as pd

# Build a tiny dataset with unordered and ordered categories.
shirt_data = pd.DataFrame(
    {"color": ["red", "blue", "red", "green"], "size": ["M", "S", "L", "M"]}
)

# Mark color as unordered and size as ordered.
shirt_data["color"] = shirt_data["color"].astype("category")
size_type = pd.CategoricalDtype(categories=["S", "M", "L"], ordered=True)
shirt_data["size"] = shirt_data["size"].astype(size_type)

# Validate that both columns are categorical.
if str(shirt_data["color"].dtype) != "category":
    raise ValueError("The color column should be categorical.")

# Display the visible labels and the category metadata.
print("Data with categorical dtypes:")
print(shirt_data.head(4).to_string(index=False))
print("Color categories:", list(shirt_data["color"].cat.categories))
print("Size categories:", list(shirt_data["size"].cat.categories))
print("Is size ordered?", shirt_data["size"].cat.ordered)

# Show that pandas stores compact integer codes internally.
print("Color internal codes:", shirt_data["color"].cat.codes.tolist())
print("Size internal codes:", shirt_data["size"].cat.codes.tolist())
print("Use explicit encoders later; these codes are not final model features.")



## **2. Category Handling**

### **2.1. Unknown Category Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_01.jpg?v=1783982389" width="250">



>* Training categories may miss future values
>* Plan handling to keep predictions reliable

>* One-hot unknowns can use all-zero indicators
>* Ordinal unknowns need reserved placeholder codes

>* Separate true new categories from data errors
>* Monitor unknowns to detect model drift



In [ ]:
#@title Python Code - Unknown Category Handling

# This example handles unseen categorical labels.
# One-hot and ordinal encoders behave differently.
# The output shows safe unknown-category codes.

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

# Training data contains only categories seen before deployment.
train_data = pd.DataFrame(
    {"region": ["North", "South", "East", "West"], "plan": ["Basic", "Plus", "Basic", "Premium"]}
)

# New data includes categories absent from the training data.
new_data = pd.DataFrame(
    {"region": ["Central", "North"], "plan": ["Plus", "Enterprise"]}
)

if train_data.shape != (4, 2) or new_data.shape != (2, 2):
    raise ValueError("Expected two small categorical tables.")

# One-hot encoding can ignore unknowns by producing all zeros.
one_hot = ColumnTransformer(
    [("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["region", "plan"])]
)

one_hot.fit(train_data)
one_hot_result = one_hot.transform(new_data)
one_hot_names = one_hot.get_feature_names_out()

# Ordinal encoding uses a reserved code for unknown categories.
ordinal = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
ordinal.fit(train_data)
ordinal_result = ordinal.transform(new_data)

print("One-hot feature names:", list(one_hot_names))
print("Central/Plus one-hot:", one_hot_result[0].astype(int).tolist())
print("North/Enterprise one-hot:", one_hot_result[1].astype(int).tolist())
print("Ordinal categories:", [list(values) for values in ordinal.categories_])
print("Ordinal encoded new rows:", ordinal_result.astype(int).tolist())
print("Unknown ordinal categories use -1 instead of a learned code.")



### **2.2. Infrequent Category Grouping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_02.jpg?v=1783982387" width="250">



>* Rare categories create large, sparse encodings
>* Group them as infrequent to simplify features

>* Improves model stability and generalization
>* Reduces overfitting to rare category noise

>* Choose thresholds based on data and goals
>* Learn grouping on training data only



In [ ]:
#@title Python Code - Infrequent Category Grouping

# This example groups rare categorical values.
# OneHotEncoder learns grouping from training data.
# Unknown values join the infrequent category.

import pandas as pd
import sklearn
from sklearn.preprocessing import OneHotEncoder

# Training data contains common and rare product brands.
train_data = pd.DataFrame(
    {"brand": ["A", "A", "A", "B", "B", "C", "D", "E"]}
)

# New data includes a known rare brand and an unseen brand.
new_data = pd.DataFrame({"brand": ["A", "C", "Z"]})

# The encoder groups categories seen fewer than two times.
encoder = OneHotEncoder(
    min_frequency=2,
    handle_unknown="infrequent_if_exist",
    sparse_output=False,
)

# Fit only on training data to avoid leakage.
encoder.fit(train_data)

if train_data.shape[0] != 8:
    raise ValueError("Expected eight training rows for this lesson.")

# Transform both training and new examples consistently.
encoded_new = encoder.transform(new_data)
encoded_table = pd.DataFrame(
    encoded_new,
    columns=encoder.get_feature_names_out(["brand"]),
)

print("scikit-learn version:", sklearn.__version__)
print("Learned categories:", list(encoder.categories_[0]))
print("Output columns:", list(encoded_table.columns))
print("New brands:", list(new_data["brand"]))
print(encoded_table.astype(int).to_string(index=False))



### **2.3. Drop Options**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_03.jpg?v=1783982391" width="250">



>* Drop categories to remove redundant indicators
>* Omitted categories become interpretation baselines

>* Dropped categories define the comparison baseline
>* Choose meaningful defaults, not arbitrary ordering

>* Coordinate drops with rare category grouping
>* Avoid all-zero ambiguity for unknown categories



In [ ]:
#@title Python Code - Drop Options

# This example compares one-hot drop options.
# Dropping creates a baseline reference category.
# Unknown categories can share all-zero encodings.

import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import sklearn

# Training data has three known payment categories.
train = pd.DataFrame({"payment": ["card", "cash", "debit", "card"]})
new = pd.DataFrame({"payment": ["card", "cash", "crypto"]})

# The encoder drops the first category alphabetically.
encoder = OneHotEncoder(
    drop="first", handle_unknown="ignore", sparse_output=False
)
encoded_train = encoder.fit_transform(train)
encoded_new = encoder.transform(new)

# Validate the small teaching example before displaying results.
if encoded_new.shape != (3, 2):
    raise ValueError("Expected three rows and two encoded columns.")

# Build compact tables for beginner-friendly output.
columns = encoder.get_feature_names_out(["payment"])
train_table = pd.DataFrame(encoded_train, columns=columns).astype(int)
new_table = pd.DataFrame(encoded_new, columns=columns).astype(int)

print("scikit-learn version:", sklearn.__version__)
print("Categories learned:", list(encoder.categories_[0]))
print("Dropped baseline category:", encoder.categories_[0][0])
print("Training encodings:", train_table.to_dict(orient="records"))
print("New encodings:", new_table.to_dict(orient="records"))
print("Notice: cash and unknown crypto both become all zeros here.")



## **3. Encoder Pipelines**

### **3.1. Target Encoding**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_01.jpg?v=1783982393" width="250">



>* Replace categories with target-based numeric summaries
>* Useful for high-cardinality pipeline transformations

>* Prevent target leakage with careful encoding
>* Use cross-validation and smoothing for reliability

>* Route each column to suitable preprocessing
>* Improve reproducible, leak-resistant model evaluation



In [ ]:
#@title Python Code - Target Encoding

# This example demonstrates target encoding in pipelines.
# ColumnTransformer routes different columns to encoders.
# Cross-validation evaluates preprocessing without target leakage.

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.linear_model import LogisticRegression

# Build a small dataset with mixed feature types.
rng = np.random.default_rng(42)
n_rows = 120

# High-cardinality stores are suitable for target encoding.
stores = np.array(["S01", "S02", "S03", "S04", "S05", "S06"])
store = rng.choice(stores, size=n_rows)

# Low-cardinality regions are suitable for one-hot encoding.
region = rng.choice(["North", "South", "West"], size=n_rows)

# Create a target related to store and region.
store_risk = {"S01": 0.10, "S02": 0.20, "S03": 0.35}
store_risk.update({"S04": 0.55, "S05": 0.70, "S06": 0.85})
region_bonus = {"North": -0.05, "South": 0.00, "West": 0.05}

# Convert category effects into churn probabilities.
probability = np.array([store_risk[value] for value in store])
probability = probability + np.array([region_bonus[value] for value in region])
probability = np.clip(probability, 0.05, 0.95)

# Sample the binary target deterministically.
churned = rng.binomial(1, probability)
data = pd.DataFrame({"store": store, "region": region})

# Validate the small teaching dataset before modeling.
if len(data) != len(churned):
    raise ValueError("Feature rows and target values must match.")

# Route each categorical column to an appropriate encoder.
preprocess = ColumnTransformer(
    transformers=[
        ("target_store", TargetEncoder(random_state=42), ["store"]),
        ("onehot_region", OneHotEncoder(handle_unknown="ignore"), ["region"]),
    ]
)

# Keep preprocessing and modeling inside one pipeline.
model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(max_iter=200, random_state=42)),
    ]
)

# Cross-validation refits encoders inside each training fold.
scores = cross_val_score(model, data, churned, cv=5, scoring="accuracy")
mean_score = round(float(scores.mean()), 3)

# Fit once to inspect the learned transformed feature names.
model.fit(data, churned)
feature_names = model.named_steps["preprocess"].get_feature_names_out()

print("scikit-learn version:", sklearn.__version__)
print("Mean cross-validated accuracy:", mean_score)
print("Transformed feature count:", len(feature_names))
print("First transformed features:", list(feature_names[:4]))



### **3.2. Sparse Encoder Output**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_02.jpg?v=1783982395" width="250">



>* One-hot encoding creates many mostly-zero columns
>* Sparse output saves memory and improves scalability

>* Sparse pipelines save memory through training
>* Check downstream sparse data compatibility

>* ColumnTransformer may mix sparse and dense features
>* Sparse choices affect compatibility and scalability



In [ ]:
#@title Python Code - Sparse Encoder Output

# This example shows sparse encoder output.
# ColumnTransformer combines categorical and numeric features.
# The result compares sparse and dense memory.

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# A small table mimics high-cardinality categorical data.
row_count = 200
postal_codes = ["P" + str(number) for number in range(100)]

# Each row has one postal code and one dense numeric value.
data = pd.DataFrame(
    {
        "postal_code": postal_codes + postal_codes,
        "age": np.linspace(20, 80, row_count),
    }
)

# ColumnTransformer keeps sparse output when the combined result is sparse.
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(sparse_output=True), ["postal_code"]),
        ("num", StandardScaler(), ["age"]),
    ],
    sparse_threshold=1.0,
)

# Fit and transform the mixed feature table.
transformed = preprocessor.fit_transform(data)

# Validate the expected sparse matrix behavior.
if not hasattr(transformed, "nnz"):
    raise RuntimeError("Expected a sparse matrix from the transformer.")

# Compare stored values with a dense version of the same matrix.
dense_version = transformed.toarray()
sparse_values = transformed.nnz
all_values = dense_version.size

# Estimate memory used by sparse data arrays and dense arrays.
sparse_bytes = transformed.data.nbytes + transformed.indices.nbytes
sparse_bytes = sparse_bytes + transformed.indptr.nbytes
dense_bytes = dense_version.nbytes

print("scikit-learn version:", sklearn.__version__)
print("Transformed shape:", transformed.shape)
print("Output type:", type(transformed).__name__)
print("Nonzero stored values:", sparse_values, "of", all_values)
print("Approx sparse memory KB:", round(sparse_bytes / 1024, 1))
print("Approx dense memory KB:", round(dense_bytes / 1024, 1))



### **3.3. ColumnTransformer Workflows**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_03.jpg?v=1783982397" width="250">



>* Apply tailored preprocessing to different column groups
>* Process categorical and numeric features in parallel

>* Keep preprocessing consistent across datasets
>* Preserve learned category structures during prediction

>* Refit encoders during cross-validation to prevent leakage
>* Explicit preprocessing improves maintenance and deployment



In [ ]:
#@title Python Code - ColumnTransformer Workflows

# Build one preprocessing workflow for mixed columns.
# Route categorical and numeric features separately.
# Show stable columns for new categories.

import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Create a small mixed-type customer dataset.
data = pd.DataFrame(
    {
        "region": ["North", "South", "East", "West", "North", "East", "South", "West"],
        "plan": ["Basic", "Pro", "Basic", "Premium", "Pro", "Premium", "Basic", "Pro"],
        "monthly_spend": [29, 79, 35, 120, 85, 130, 40, 90],
        "account_months": [3, 18, 5, 30, 20, 36, 7, 24],
        "renewed": [0, 1, 0, 1, 1, 1, 0, 1],
    }
)

# Separate features from the target label.
features = data.drop(columns="renewed")
target = data["renewed"]

# Validate the expected beginner-friendly table shape.
if features.shape != (8, 4):
    raise ValueError("Expected 8 rows and 4 feature columns.")

# Split before fitting to avoid preprocessing leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
    stratify=target,
)

# Name columns by the preprocessing they need.
categorical_columns = ["region", "plan"]
numeric_columns = ["monthly_spend", "account_months"]

# Build one transformer with separate column routes.
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("numeric", StandardScaler(), numeric_columns),
    ]
)

# Put preprocessing and modeling into one reusable pipeline.
model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(random_state=42, max_iter=200)),
    ]
)

# Fit the whole workflow only on training data.
model.fit(X_train, y_train)

# Transform new data containing an unseen category safely.
new_customers = pd.DataFrame(
    {
        "region": ["Central", "North"],
        "plan": ["Pro", "Basic"],
        "monthly_spend": [75, 32],
        "account_months": [12, 4],
    }
)

# Inspect the learned feature names after preprocessing.
feature_names = model.named_steps["preprocess"].get_feature_names_out()
new_matrix = model.named_steps["preprocess"].transform(new_customers)

# Predict with the same learned preprocessing structure.
predictions = model.predict(new_customers)
accuracy = model.score(X_test, y_test)

print("scikit-learn version:", sklearn.__version__)
print("Transformed feature count:", len(feature_names))
print("First five learned features:", list(feature_names[:5]))
print("New data transformed shape:", new_matrix.shape)
print("Test accuracy:", round(accuracy, 2))
print("Predictions for new customers:", list(predictions))



# <font color="#418FDE" size="6.5" uppercase>**Categorical Encoding**</font>


In this lecture, you learned to:
- Encode categorical features using ordinal and one-hot strategies. 
- Handle unknown, infrequent, and redundant categories in practical datasets. 
- Integrate categorical encoders into ColumnTransformer workflows. 

In the next Lecture (Lecture B), we will go over 'Imputation Targets'